In [1]:
# pp.ipynb - preprocess count data and save into formatted adata files.

In [2]:
import anndata as ad
import numpy as np
import os
import pandas as pd
import scanpy as sc

In [3]:
starsolo_fn = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/base/gen_data/sccnasim/rdr_starsolo/HCC3N_simu.spotxgene.starsolo.h5ad"
afc_fn = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/base/gen_data/sccnasim/baf_afc/afc.counts.cell_anno.h5ad"

out_dir = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/internal_consistency/simu_tools/pp"

In [4]:
normal_cell_type = "normal"
tumor_cell_type = "tumor"

# Load Data

In [5]:
os.makedirs(out_dir, exist_ok = True)

### STARsolo

In [6]:
starsolo = ad.read_h5ad(starsolo_fn)
starsolo

/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


AnnData object with n_obs × n_vars = 1200 × 33538
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand'

In [7]:
starsolo.var['feature'] = starsolo.var['gene']
starsolo = starsolo[:, ~starsolo.var["feature"].duplicated(keep = False)].copy()
starsolo.obs.index = starsolo.obs["cell"]
starsolo.var.index = starsolo.var["feature"]
starsolo

AnnData object with n_obs × n_vars = 1200 × 33490
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand', 'feature'

### scCNASim afc

In [8]:
afc = ad.read_h5ad(afc_fn)
afc

/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


AnnData object with n_obs × n_vars = 1200 × 32295
    obs: 'cell', 'cell_type'
    var: 'chrom', 'start', 'end', 'feature', 'strand'
    layers: 'A', 'B', 'U'

In [9]:
afc = afc[:, ~afc.var["feature"].duplicated(keep = False)].copy()
afc.X = afc.layers["A"] + afc.layers["B"] + afc.layers["U"]
afc.obs.index = afc.obs["cell"]
afc.var.index = afc.var["feature"]
afc

AnnData object with n_obs × n_vars = 1200 × 32295
    obs: 'cell', 'cell_type'
    var: 'chrom', 'start', 'end', 'feature', 'strand'
    layers: 'A', 'B', 'U'

### Use the same order of cells

In [10]:
starsolo = starsolo[afc.obs.index, :].copy()
starsolo

AnnData object with n_obs × n_vars = 1200 × 33490
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand', 'feature'

## Use intersect genes

In [11]:
adata_list = [starsolo, afc]

genes = None
for i, adata in enumerate(adata_list):
    if i == 0:
        genes = adata.var["feature"].to_numpy()
    else:
        genes = np.intersect1d(genes, adata.var["feature"])
genes.shape

(32272,)

### Use the same order of genes

In [12]:
afc = afc[:, afc.var["feature"].isin(genes)].copy()
print(afc)

starsolo = starsolo[:, afc.var.index].copy()
print(starsolo)

AnnData object with n_obs × n_vars = 1200 × 32272
    obs: 'cell', 'cell_type'
    var: 'chrom', 'start', 'end', 'feature', 'strand'
    layers: 'A', 'B', 'U'
AnnData object with n_obs × n_vars = 1200 × 32272
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand', 'feature'


## Split by cell type

In [13]:
starsolo_normal = starsolo[starsolo.obs["cell_type"] == normal_cell_type, :].copy()
print(starsolo_normal)

starsolo_tumor = starsolo[starsolo.obs["cell_type"] == tumor_cell_type, :].copy()
print(starsolo_tumor)

afc_normal = afc[afc.obs["cell"].isin(starsolo_normal.obs["cell"]), :].copy()
print(afc_normal)

afc_tumor = afc[afc.obs["cell"].isin(starsolo_tumor.obs["cell"]), :].copy()
print(afc_tumor)

AnnData object with n_obs × n_vars = 600 × 32272
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand', 'feature'
AnnData object with n_obs × n_vars = 600 × 32272
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand', 'feature'
AnnData object with n_obs × n_vars = 600 × 32272
    obs: 'cell', 'cell_type'
    var: 'chrom', 'start', 'end', 'feature', 'strand'
    layers: 'A', 'B', 'U'
AnnData object with n_obs × n_vars = 600 × 32272
    obs: 'cell', 'cell_type'
    var: 'chrom', 'start', 'end', 'feature', 'strand'
    layers: 'A', 'B', 'U'


# Save Data

### Check the order of cells and genes

In [14]:
assert np.all(starsolo_normal.obs["cell"].to_numpy() == afc_normal.obs["cell"].to_numpy())
assert np.all(starsolo_tumor.obs["cell"].to_numpy() == afc_tumor.obs["cell"].to_numpy())

In [15]:
assert np.all(starsolo_normal.var["feature"].to_numpy() == afc_normal.var["feature"].to_numpy())
assert np.all(starsolo_tumor.var["feature"].to_numpy() == afc_tumor.var["feature"].to_numpy())

In [16]:
starsolo_normal.var.index.name = None
starsolo_tumor.var.index.name = None
afc_normal.var.index.name = None
afc_tumor.var.index.name = None

In [17]:
starsolo_normal.write_h5ad(os.path.join(out_dir, "starsolo_normal.h5ad"), compression = 'gzip')
starsolo_tumor.write_h5ad(os.path.join(out_dir, "starsolo_tumor.h5ad"), compression = 'gzip')
afc_normal.write_h5ad(os.path.join(out_dir, "afc_normal.h5ad"), compression = 'gzip')
afc_tumor.write_h5ad(os.path.join(out_dir, "afc_tumor.h5ad"), compression = 'gzip')